# Forex Card Enhancement — Exploratory Data Analysis
**Analyst:** Sagar Kandelkar | **Date:** September 2026
**Data:** Synthetic Forex card dataset for portfolio case study

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)

## 1. Load Data

In [ ]:
customers = pd.read_csv('../data/forex_customers.csv')
cards = pd.read_csv('../data/forex_cards.csv')
transactions = pd.read_csv('../data/forex_transactions.csv')
fx_rates = pd.read_csv('../data/fx_rates.csv')

print('Customers:', customers.shape)
print('Cards:', cards.shape)
print('Transactions:', transactions.shape)
print('FX Rates:', fx_rates.shape)

## 2. Card Status Distribution

In [ ]:
status_counts = cards['status'].value_counts()
colors = ['#2563eb', '#dc2626', '#f59e0b']
status_counts.plot(kind='bar', color=colors)
plt.title('Card Status Distribution')
plt.xlabel('Status')
plt.ylabel('Count')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()
print(status_counts)

## 3. LRS Utilization Analysis

In [ ]:
customers['utilization_pct'] = (customers['lrs_used_usd'] / customers['lrs_limit_usd']) * 100

fig, ax = plt.subplots()
sns.histplot(customers['utilization_pct'], bins=10, kde=True, color='#2563eb', ax=ax)
ax.axvline(customers['utilization_pct'].mean(), color='#dc2626', linestyle='--', label=f'Mean: {customers["utilization_pct"].mean():.1f}%')
ax.set_title('LRS Utilization Distribution')
ax.set_xlabel('Utilization %')
ax.legend()
plt.tight_layout()
plt.show()

print(f'Average LRS utilization: {customers["utilization_pct"].mean():.1f}%')
print(f'Median LRS utilization: {customers["utilization_pct"].median():.1f}%')

## 4. Transaction Volume by Currency

In [ ]:
currency_vol = transactions.groupby('transaction_currency')['local_currency_amount'].sum().sort_values(ascending=False)
currency_vol.plot(kind='bar', color='#2563eb')
plt.title('Transaction Volume by Currency (INR)')
plt.xlabel('Currency')
plt.ylabel('Total INR')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

## 5. Transaction Type Split

In [ ]:
type_split = transactions.groupby('transaction_type')['local_currency_amount'].agg(['count', 'sum'])
type_split.columns = ['Transaction Count', 'Total INR']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
type_split['Transaction Count'].plot(kind='pie', ax=axes[0], autopct='%1.1f%%', colors=['#2563eb', '#dc2626', '#f59e0b'])
axes[0].set_title('Transaction Count by Type')
axes[0].set_ylabel('')

type_split['Total INR'].plot(kind='pie', ax=axes[1], autopct='%1.1f%%', colors=['#2563eb', '#dc2626', '#f59e0b'])
axes[1].set_title('Volume by Transaction Type')
axes[1].set_ylabel('')

plt.tight_layout()
plt.show()
print(type_split)

## 6. Monthly Transaction Trend

In [ ]:
transactions['transaction_date'] = pd.to_datetime(transactions['transaction_date'])
transactions['month'] = transactions['transaction_date'].dt.to_period('M').astype(str)

monthly = transactions.groupby('month').agg({'transaction_id': 'count', 'local_currency_amount': 'sum'}).reset_index()
monthly.columns = ['Month', 'Count', 'Volume']

fig, ax1 = plt.subplots()
ax1.bar(monthly['Month'], monthly['Count'], color='#2563eb', alpha=0.7, label='Count')
ax1.set_xlabel('Month')
ax1.set_ylabel('Transaction Count', color='#2563eb')
ax1.tick_params(axis='y', labelcolor='#2563eb')
ax1.set_xticklabels(monthly['Month'], rotation=45)

ax2 = ax1.twinx()
ax2.plot(monthly['Month'], monthly['Volume'], color='#dc2626', marker='o', linewidth=2, label='Volume')
ax2.set_ylabel('Volume (INR)', color='#dc2626')
ax2.tick_params(axis='y', labelcolor='#dc2626')

plt.title('Monthly Transaction Trends')
plt.tight_layout()
plt.show()

## 7. Card Spend Rate Analysis

In [ ]:
cards['spend_rate'] = (cards['total_spent_inr'] / cards['total_loaded_inr']) * 100
active_cards = cards[cards['status'] == 'active']

sns.histplot(active_cards['spend_rate'], bins=8, kde=True, color='#2563eb')
plt.title('Card Spend Rate Distribution (Active Cards)')
plt.xlabel('Spend Rate %')
plt.ylabel('Count')
plt.tight_layout()
plt.show()

print(f'Average spend rate: {active_cards["spend_rate"].mean():.1f}%')
print(f'Median spend rate: {active_cards["spend_rate"].median():.1f}%')

## 8. Top Merchant Countries

In [ ]:
country_vol = transactions.groupby('merchant_country')['local_currency_amount'].sum().sort_values(ascending=True)
country_vol.plot(kind='barh', color='#2563eb')
plt.title('Transaction Volume by Country')
plt.xlabel('Total INR')
plt.tight_layout()
plt.show()

## 9. Key Insights & Recommendations

1. **LRS Utilization:** Average utilization is moderate (~38%), with some customers nearing limits. Proactive alerts recommended.
2. **Currency Dominance:** USD and GBP account for the highest transaction volumes — focus rate competitiveness here.
3. **Transaction Types:** POS dominates count and volume, but ecom is growing. Ensure online channel security.
4. **Card Activity:** Several cards have high spend rates (>95%) with low remaining balances — repatriation nudges needed.
5. **Geography:** USA, UK, and UAE are top destinations — tailor marketing and rate offers.
6. **Blocked Cards:** 10% of cards are blocked — investigate root causes (expired KYC, lost cards).

### Recommendations
- Implement LRS limit alerts at 70%, 85%, and 95% thresholds
- Offer rate-lock promotions for USD and GBP during peak travel seasons
- Push repatriation reminders when card balance drops below ₹5,000 and expiry is <60 days
- Enable instant digital reactivation for KYC-pending cards
- Add travel insurance bundling for high-value card users